# Python Sales Data Analysis

**Portfolio project | Bester Malama | Lusaka, Zambia**

Analyze 20 fictional retail sales orders from September 2026 using **Python, Pandas, and Matplotlib**. This notebook loads its dataset directly from the public GitHub repository, validates sales figures, and generates business visualizations.

**How to run:** In Google Colab choose **Runtime → Run all**. Internet access is required to load the CSV from GitHub. No manual Excel upload is necessary.

## 1. Load and inspect the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/bestermalam/Python-Sales-Data-Analysis/main/Cleaned_Sales_Data.csv"

df = pd.read_csv(DATA_URL, parse_dates=["Date"])
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data quality checks

Validate order identifiers, missing data, duplicate orders, and calculated line totals. The CSV is already a cleaned dataset; these checks protect against accidental mistakes when it is reused.

In [ ]:
required = ["Order ID", "Date", "Product", "Category", "City", "Salesperson", "Quantity", "Unit Price", "Total Sales"]
assert all(col in df.columns for col in required), "A required column is missing."
print("Missing values per column:")
display(df[required].isna().sum().to_frame("Missing"))
print("Duplicated order IDs:", df["Order ID"].duplicated().sum())

valid_order = df["Order ID"].astype(str).str.fullmatch(r"ORD\d+")
df_clean = df.loc[valid_order].drop_duplicates(subset=["Order ID"]).copy()

line_total_difference = (df_clean["Quantity"] * df_clean["Unit Price"] - df_clean["Total Sales"]).abs()
print("Orders with quantity × unit price mismatch:", (line_total_difference > 0.01).sum())
if (line_total_difference > 0.01).any():
    display(df_clean.loc[line_total_difference > 0.01, ["Order ID", "Product", "Quantity", "Unit Price", "Total Sales"]])

**Interpretation:** An order amount that differs from quantity × unit price should be investigated. It may reflect a data-entry error, discount, or a pricing change. This notebook retains the recorded `Total Sales` field for consistency with the project dashboard.

## 3. Key performance indicators

In [ ]:
kpis = pd.Series({
    "Total revenue (ZMW)": df_clean["Total Sales"].sum(),
    "Total orders": df_clean["Order ID"].nunique(),
    "Average order value (ZMW)": df_clean["Total Sales"].mean(),
    "Highest order (ZMW)": df_clean["Total Sales"].max(),
    "Lowest order (ZMW)": df_clean["Total Sales"].min(),
})
display(kpis.to_frame("Value"))
assert len(df_clean) == 20, "Unexpected number of orders; review source data."
assert abs(df_clean["Total Sales"].sum() - 120560) < 0.01, "Revenue differs from verified Excel dashboard."

## 4. Sales performance by city

In [ ]:
sales_by_city = df_clean.groupby("City")["Total Sales"].sum().sort_values(ascending=False)
display(sales_by_city.to_frame("Revenue (ZMW)"))
plt.figure(figsize=(9, 5))
plt.bar(sales_by_city.index, sales_by_city.values)
plt.title("Sales Performance by City")
plt.xlabel("City")
plt.ylabel("Total Revenue (ZMW)")
plt.tight_layout()
plt.show()

## 5. Sales performance by product

In [ ]:
sales_by_product = df_clean.groupby("Product")["Total Sales"].sum().sort_values(ascending=True)
display(sales_by_product.sort_values(ascending=False).to_frame("Revenue (ZMW)"))
plt.figure(figsize=(10, 6))
plt.barh(sales_by_product.index, sales_by_product.values)
plt.title("Sales Performance by Product")
plt.xlabel("Total Revenue (ZMW)")
plt.tight_layout()
plt.show()

## 6. Sales performance by category

In [ ]:
sales_by_category = df_clean.groupby("Category")["Total Sales"].sum().sort_values(ascending=False)
display(sales_by_category.to_frame("Revenue (ZMW)"))
plt.figure(figsize=(9, 5))
plt.bar(sales_by_category.index, sales_by_category.values)
plt.title("Sales Performance by Category")
plt.xlabel("Category")
plt.ylabel("Total Revenue (ZMW)")
plt.tight_layout()
plt.show()

## 7. Salesperson performance

In [ ]:
sales_by_person = df_clean.groupby("Salesperson")["Total Sales"].sum().sort_values(ascending=False)
display(sales_by_person.to_frame("Revenue (ZMW)"))
plt.figure(figsize=(9, 5))
plt.bar(sales_by_person.index, sales_by_person.values)
plt.title("Sales Performance by Salesperson")
plt.xlabel("Salesperson")
plt.ylabel("Total Revenue (ZMW)")
plt.tight_layout()
plt.show()

## 8. Daily sales trend

In [ ]:
daily_sales = df_clean.groupby("Date")["Total Sales"].sum().sort_index()
display(daily_sales.to_frame("Revenue (ZMW)"))
plt.figure(figsize=(12, 5))
plt.plot(daily_sales.index, daily_sales.values, marker="o", linewidth=2)
plt.title("Daily Sales Revenue Trend — September 2026")
plt.xlabel("Date")
plt.ylabel("Total Revenue (ZMW)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 9. Business findings

- Total revenue across the 20 verified transactions: **K120,560**.
- **Electronics** contributed **K87,700** (approximately **72.7%** of sales revenue).
- **John** generated the highest salesperson revenue (**K47,200**).
- **Tablets** generated the most product revenue (**K22,500**), followed by **phones** (**K22,400**).

**Limitations:** This is a small practice dataset. Revenue is not the same as profit, and these results should not be generalized to a real business without additional data.

## 10. Export cleaned transactions (optional)

In [ ]:
df_clean.to_csv("Cleaned_Sales_Data.csv", index=False)
print("Exported", len(df_clean), "verified orders to Cleaned_Sales_Data.csv")

## Project links

[GitHub project repository](https://github.com/bestermalam/Python-Sales-Data-Analysis)